# KV-cache compression: minimal starter

Four code cells: dependencies, model, context/cache inspection, and LM Evaluation Harness.
Use a fresh Python 3.10+ notebook kernel; a Colab GPU runtime is recommended.
Run top to bottom. Model weights and benchmark data download on first use.

This is an **uncompressed baseline and inspection scaffold**. ILF selection,
mixed-precision cache storage, and their integration into inference are left for you to implement.
The default 0.5B model and 5-question evaluation are for checking the pipeline.

## 1. Dependencies

Transformers and the harness are pinned to keep the cache API stable. If installing
changes a package already imported in your kernel, restart the kernel before continuing.

In [ ]:
%pip install -q "torch>=2.6" "transformers==4.57.1" "accelerate>=0.26.0" "lm_eval[hf,longbench]==0.4.11" sentencepiece

import json
from pathlib import Path
from importlib.metadata import version

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from lm_eval import evaluator
from lm_eval.models.huggingface import HFLM
from lm_eval.utils import make_table

set_seed(42)
print({name: version(name) for name in ["torch", "transformers", "lm_eval"]})


## 2. Load the model

Change `MODEL_ID` to a larger Qwen2.5 or a Llama model later. Gated Llama
checkpoints require Hugging Face access and login. Eager attention lets us inspect
attention weights; keep the same backend across comparisons.

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-0.5B"  # Small, public base model for a first run.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = (
    torch.bfloat16 if DEVICE == "cuda" and torch.cuda.is_bf16_supported()
    else torch.float16 if DEVICE == "cuda"
    else torch.float32
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, padding_side="left")
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=DTYPE,
    attn_implementation="eager",
).to(DEVICE).eval()

print(f"{MODEL_ID} | {model.device} | {model.dtype}")


## 3. Expose the context: token IDs, KV cache, and attention weights

The text context becomes `inputs.input_ids`. A prefill forward pass with
`use_cache=True` returns its KV cache in `past_key_values`.
Each layer stores K/V as **[batch, KV heads, cached tokens, head dimension]**.
Attention weights are **[batch, query heads, query tokens, key tokens]**.

Keep this example short: returning every attention matrix uses quadratic memory.
`attention_scores[l, i]` below is an illustrative importance score for token `i`
in layer `l`, averaged over heads and the final 8 query positions. This pooling
choice is a starting point for your experiments, not a fixed part of the proposal.

In [ ]:
CONTEXT = (
    "The archive has three rooms. The red room holds maps. "
    "The blue room holds a brass key. The green room holds letters. "
    "Question: Which room holds the brass key? Answer:"
)
inputs = tokenizer(
    CONTEXT, return_tensors="pt", truncation=True, max_length=256
).to(DEVICE)

with torch.inference_mode():
    prefill = model(
        **inputs, use_cache=True, output_attentions=True, return_dict=True
    )

kv_cache = prefill.past_key_values  # Transformers DynamicCache, across layers.
attentions = prefill.attentions   # Tuple: one attention tensor per layer.
keys = kv_cache.layers[0].keys    # Inspect/edit layer 0; repeat for other layers.
values = kv_cache.layers[0].values
attention_scores = torch.stack([
    weights[0, :, -8:, :].float().mean(dim=(0, 1))
    for weights in attentions
])  # [num_layers, num_context_tokens]; batch size is 1 here.

kv_bytes = sum(
    tensor.numel() * tensor.element_size()
    for layer in kv_cache.layers
    for tensor in (layer.keys, layer.values)
)
print("Input tokens:", inputs.input_ids.shape[-1])
print("Layer 0 K/V:", keys.shape, values.shape)
print("Attention scores [layers, tokens]:", attention_scores.shape)
print(f"Uncompressed KV tensors: {kv_bytes / 2**20:.3f} MiB")

# TODO: choose per-layer thresholds tau; compute ILF and diversified scores.
# TODO: use those scores to select tokens and/or allocate 0/4/8/16 bits.


## 4. Benchmark the baseline with LM Evaluation Harness

This wraps the **same loaded model**, so it does not load a second copy.
Start with 5 questions from one MMLU subject. For full MMLU, set
`TASKS = ["mmlu"]`, `LIMIT = None`, and `NUM_FEWSHOT = 5`;
the limit applies separately to every subject.

**Editing `kv_cache` above does not change this benchmark.** MMLU scores answer
likelihoods using forward passes; it does not test a compressed decode loop.
To evaluate compression, integrate it into the model/cache implementation or
a custom `HFLM` adapter, handling both likelihood scoring and generation as needed.
Token eviction also needs correct attention masks and original positional indices.
Integer bit allocation alone does not provide packed low-bit storage.

LongBench and separate GPU memory/decode-latency measurements are next steps.
The inspection cell's KV tensor byte count excludes model weights and allocator
overhead; harness runtime is not per-token decode latency. For the proposal's
memory/latency comparison, use a separate batch-4, 4096-token workload, with
attention diagnostics off and CUDA synchronization around timing.

In [ ]:
TASKS = ["mmlu_abstract_algebra"]
LIMIT = 5           # Smoke test only; None evaluates every example.
NUM_FEWSHOT = 0     # Use 5 for the usual five-shot MMLU setup.
BATCH_SIZE = 1

lm = HFLM(pretrained=model, tokenizer=tokenizer, batch_size=BATCH_SIZE)
results = evaluator.simple_evaluate(
    model=lm,
    tasks=TASKS,
    num_fewshot=NUM_FEWSHOT,
    limit=LIMIT,
    apply_chat_template=False,  # Base model; no chat template.
    log_samples=False,
    random_seed=42,
    numpy_random_seed=42,
    torch_random_seed=42,
    fewshot_random_seed=42,
)
print(make_table(results))

output_path = Path("results/mmlu_baseline.json")
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(json.dumps({
    "model_id": MODEL_ID,
    "model_revision": getattr(model.config, "_commit_hash", None),
    "compression": "none",
    "dtype": str(model.dtype),
    "device": str(model.device),
    "packages": {name: version(name) for name in ["torch", "transformers", "lm_eval"]},
    "evaluation": results,
}, indent=2, default=str), encoding="utf-8")
print(f"Saved {output_path}")


References: [Hugging Face cache guide](https://huggingface.co/docs/transformers/v4.57.1/en/cache_explanation),
[LM Evaluation Harness Python API](https://github.com/EleutherAI/lm-evaluation-harness/blob/main/docs/python-api.md),
[LongBench tasks](https://github.com/EleutherAI/lm-evaluation-harness/blob/main/lm_eval/tasks/longbench/README.md),
[Qwen2.5-0.5B model card](https://huggingface.co/Qwen/Qwen2.5-0.5B).